# TriageIQ - 02 Baseline Classifier
TF-IDF + Logistic Regression, time-based split, macro-F1.

Run after `01_eda`. Needs `../data/311_requests.csv`.

## 1. Load and prepare

In [ ]:
import pandas as pd, numpy as np, joblib, matplotlib.pyplot as plt
df = pd.read_csv("../data/311_requests.csv", parse_dates=["created_date"])
df["description"] = df["description"].fillna("").str.lower().str.strip()
df["neighborhood"] = df["neighborhood"].fillna("Unknown")
df["hour"] = df.created_date.dt.hour.astype(str)
df["weekday"] = df.created_date.dt.weekday.astype(str)
df.shape

## 2. Time-based split (train before 2025-10-01, test after)

In [ ]:
cut = pd.Timestamp("2025-10-01")
train, test = df[df.created_date < cut], df[df.created_date >= cut]
TARGET = "category"
y_train, y_test = train[TARGET], test[TARGET]
print(len(train), len(test))

## 3. Model A: text only

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report, f1_score
text_model = make_pipeline(TfidfVectorizer(ngram_range=(1,2), min_df=2),
                           LogisticRegression(max_iter=1000, class_weight="balanced"))
text_model.fit(train["description"], y_train)
pred_a = text_model.predict(test["description"])
print(classification_report(y_test, pred_a))
f1_a = f1_score(y_test, pred_a, average="macro"); print("macro-F1 (text only):", round(f1_a,4))

## 4. Model B: text + categorical features

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
prep = ColumnTransformer([
    ("txt", TfidfVectorizer(ngram_range=(1,2), min_df=2), "description"),
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["neighborhood","channel","hour","weekday"])])
full_model = make_pipeline(prep, LogisticRegression(max_iter=1000, class_weight="balanced"))
full_model.fit(train, y_train)
pred_b = full_model.predict(test)
f1_b = f1_score(y_test, pred_b, average="macro"); print("macro-F1 (text + cat):", round(f1_b,4))

## 5. Confusion matrix

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
fig, ax = plt.subplots(figsize=(8,7))
ConfusionMatrixDisplay.from_predictions(y_test, pred_b, xticks_rotation=45, ax=ax)
plt.tight_layout(); plt.show()

## 6. Top words per class

In [ ]:
vec = text_model.named_steps["tfidfvectorizer"]; clf = text_model.named_steps["logisticregression"]
names = np.array(vec.get_feature_names_out())
for i, c in enumerate(clf.classes_):
    print(c, "->", list(names[np.argsort(clf.coef_[i])[-8:][::-1]]))

## 7. Inspect errors

In [ ]:
mask = (pred_b != y_test).values
err = test[mask].assign(predicted=pred_b[mask])
print(len(err), "errors")
err[["description","category","predicted"]].head(15)

## 8. Save the model

In [ ]:
joblib.dump(full_model, "../models/baseline.joblib")
print("saved")

## 9. Findings (fill in)
- Macro-F1 text only:
- Macro-F1 text + categorical:
- Weakest class:
- Why scores are high (synthetic, templated text):
- Next: DistilBERT comparison